In [ ]:
from authlib.jose import jwt
from authlib.jose.errors import BadSignatureError, ExpiredTokenError
import secrets

# Example 1: JWT Token Creation and Verification
print("Example 1: JWT Token Operations")

try:
    # Generate a secret key for signing
    secret = secrets.token_urlsafe(32)
    
    # Create a JWT payload
    payload = {
        'user_id': 12345,
        'username': 'testuser',
        'role': 'admin',
        'exp': 1234567890  # Expiration timestamp
    }
    
    # Create and sign the JWT token
    token = jwt.encode({'alg': 'HS256'}, payload, secret)
    print(f"✓ JWT Token created: {token[:50]}...")
    
    # Verify and decode the token
    decoded = jwt.decode(token, secret)
    print(f"✓ Token decoded successfully")
    print(f"User ID: {decoded['user_id']}")
    print(f"Username: {decoded['username']}")
    print(f"Role: {decoded['role']}")
    assert dict(decoded) == payload, decoded
    assert decoded.header['alg'] == 'HS256', decoded.header
    
    # A different secret does not verify the signature
    try:
        jwt.decode(token, secrets.token_urlsafe(32))
    except BadSignatureError:
        print("✓ Token signed with another secret is rejected")
    else:
        raise AssertionError("a token verified with the wrong secret")
    
    # Claim validation: the exp timestamp above (2009) is in the past
    try:
        decoded.validate()
    except ExpiredTokenError:
        print("✓ Expired token detected by claims validation")
    else:
        raise AssertionError("an expired token passed claims validation")
    
except Exception as e:
    print(f"Error in JWT example: {e}")
    raise AssertionError(
        f"authlib example (cell 0) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 completed\n")

In [ ]:
from authlib.jose import JsonWebKey, jwt
import base64

# Example 2: JSON Web Key (JWK) Generation and Usage
print("Example 2: JWK Operations")

try:
    # Generate a symmetric key for HMAC
    jwk = JsonWebKey.generate_key('oct', 256, is_private=True)
    print("✓ JWK generated successfully")
    
    # Export the key in JWK format
    jwk_dict = jwk.as_dict()
    print(f"Key type: {jwk_dict['kty']}")
    assert jwk_dict['kty'] == 'oct', jwk_dict
    print(f"Key ID: {jwk_dict.get('kid', 'Not set')}")
    
    # Export key for different uses
    # OctKey exposes the symmetric key bytes as `raw_key`.
    raw_key = jwk.raw_key
    print(f"✓ Raw key extracted (length: {len(raw_key)} bytes)")
    assert len(raw_key) == 32, len(raw_key)  # 256 bits
    k = jwk_dict['k']
    assert base64.urlsafe_b64decode(k + '=' * (-len(k) % 4)) == raw_key  # "k" is the base64url key
    
    # Generate an RSA key pair
    rsa_jwk = JsonWebKey.generate_key('RSA', 2048, is_private=True)
    rsa_dict = rsa_jwk.as_dict()
    print(f"✓ RSA JWK generated")
    print(f"RSA key type: {rsa_dict['kty']}")
    assert rsa_dict['kty'] == 'RSA', rsa_dict
    n = rsa_dict['n']
    modulus_bits = len(base64.urlsafe_b64decode(n + '=' * (-len(n) % 4))) * 8
    print(f"RSA key size: {modulus_bits} bits")
    assert modulus_bits == 2048, modulus_bits
    
    # Sign with the private key, verify with the public JWK alone
    public_jwk = JsonWebKey.import_key(rsa_jwk.as_dict(is_private=False))
    rs_token = jwt.encode({'alg': 'RS256'}, {'sub': 'alice'}, rsa_jwk)
    assert jwt.decode(rs_token, public_jwk)['sub'] == 'alice'
    print("✓ RS256 token verified with the public key")
    
    # Test key usage flags
    print(f"RSA key usage: {rsa_dict.get('use', 'Not specified')}")
    
except Exception as e:
    print(f"Error in JWK example: {e}")
    raise AssertionError(
        f"authlib example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 completed\n")